# 🔍 Local RAG — PDF → SQL Server 2025 (Native Vector) → gpt-oss-20b

**Stack:**
- 📄 PDF parsing → `pypdf`
- 🧠 Embeddings → Gemini `gemini-embedding-001` (free API)
- 🗄️ Vector DB → `SQL Server 2025` native `VECTOR(768)` + DiskANN index
- 💬 LLM → `gpt-oss-20b` via HuggingFace Inference Providers (Groq)
- 🧠 Memory → Conversation checkpointing to SQL Server tables
- 🔄 CDC → Chunk-level Change Data Capture (SQL Server backed)
- ⏱️ Recency → Exponential decay re-ranking

---

#### ── Core Pipeline ──
- Cell 1  — Install dependencies
- Cell 2  — Config & API keys + SQL Server connection
- Cell 3  — PDF extraction
- Cell 4  — Chunking
- Cell 5  — Gemini batch embedding
- Cell 6  — SQL Server storage & retrieval (VECTOR_SEARCH)
- Cell 7  — ▶️ Run ingestion
- Cell 8  — gpt-oss-20b answer generation
- Cell 9  — Full RAG pipeline — `ask()`
- Cell 10 — ▶️ Ask a single question
- Cell 11 — ▶️ Interactive REPL loop
- Cell 12 — ▶️ Utilities

#### ── SQL Server Explorer ──
- Cell 13 — ▶️ Direct SQL Server query explorer

#### ── Conversation Memory ──
- Cell 13A — Memory Manager (SQL Server backed)
- Cell 13B — Memory-aware `ask_with_memory()`
- Cell 13C — ▶️ Interactive memory REPL
- Cell 13D — ▶️ Initialize memory session
- Cell 13E — ▶️ Ask a single question with memory
- Cell 13F — ▶️ Ask multiple questions with memory

#### ── Staleness · CDC · Recency ──
- Cell 14A — Staleness Registry + enhanced store
- Cell 14B — CDC Engine + smart ingest
- Cell 14C — Recency-weighted retrieval + `ask_weighted()`
- Cell 14D — ▶️ Test PDF generator
- Cell 14E — ▶️ Full test suite

In [ ]:
# ── Cell 0: ▶️ CLEANUP — Drop + Recreate all RAG tables ──────────────────────
# Run this before Cell 14E test suite, or any time you want a clean slate.
#
# WHY DROP+RECREATE instead of DELETE:
#   - Resets IDENTITY counters (id column) on rag_chunks and rag_turns
#   - Drops the vector index automatically as part of DROP TABLE
#   - Faster and cleaner than DELETE on large datasets
#   - Guarantees no orphaned index state
#
# This runs 3 separate autocommit connections to match SQL Server's
# requirement that CREATE VECTOR INDEX cannot run inside a transaction.

# def reset_all_tables():
#     confirm = input("⚠️  This drops and recreates ALL RAG tables. Type 'yes' to confirm: ")
#     if confirm.strip().lower() != "yes":
#         print("Cancelled."); return

#     # ── Step 1: Drop all tables (autocommit — drops vector index automatically) ──
#     print("Step 1/3  Dropping all RAG tables...")
#     conn = pyodbc.connect(SQL_CONN_STR, autocommit=True)
#     cur  = conn.cursor()
#     # Drop in FK order: child tables first
#     for tbl in ["rag_turns", "rag_sessions", "rag_chunk_registry",
#                  "rag_staleness", "rag_chunks"]:
#         cur.execute(f"DROP TABLE IF EXISTS {tbl};")
#         print(f"   Dropped {tbl}")
#     conn.close()

#     # ── Step 2: Recreate all tables (transactional) ────────────────────────────
#     print("Step 2/3  Recreating all RAG tables...")
#     conn = get_conn()
#     cur  = conn.cursor()
#     cur.execute("""
#         CREATE TABLE rag_chunks (
#             id          INT           IDENTITY(1,1) PRIMARY KEY CLUSTERED,
#             source      NVARCHAR(500) NOT NULL,
#             chunk_index INT           NOT NULL,
#             content     NVARCHAR(MAX) NOT NULL,
#             file_hash   NVARCHAR(64),
#             ingested_at DATETIME2     DEFAULT GETUTCDATE(),
#             embedding   VECTOR(768)
#         );
#     """)
#     cur.execute("CREATE INDEX idx_rag_source ON rag_chunks(source);")
#     cur.execute("""
#         CREATE TABLE rag_staleness (
#             doc_name    NVARCHAR(500) PRIMARY KEY,
#             file_hash   NVARCHAR(64)  NOT NULL,
#             chunk_count INT,
#             ingested_at DATETIME2     DEFAULT GETUTCDATE(),
#             version     INT           DEFAULT 1
#         );
#     """)
#     cur.execute("""
#         CREATE TABLE rag_chunk_registry (
#             doc_name   NVARCHAR(500) NOT NULL,
#             chunk_hash NVARCHAR(64)  NOT NULL,
#             chunk_id   NVARCHAR(64)  NOT NULL,
#             PRIMARY KEY (doc_name, chunk_hash)
#         );
#     """)
#     cur.execute("""
#         CREATE TABLE rag_sessions (
#             session_id  NVARCHAR(100) PRIMARY KEY,
#             created_at  DATETIME2     DEFAULT GETUTCDATE(),
#             updated_at  DATETIME2     DEFAULT GETUTCDATE(),
#             model       NVARCHAR(200),
#             embed_model NVARCHAR(200),
#             turn_count  INT           DEFAULT 0
#         );
#     """)
#     cur.execute("""
#         CREATE TABLE rag_turns (
#             id         INT           IDENTITY(1,1) PRIMARY KEY,
#             session_id NVARCHAR(100) NOT NULL REFERENCES rag_sessions(session_id),
#             role       NVARCHAR(20)  NOT NULL CHECK (role IN ('user','assistant')),
#             content    NVARCHAR(MAX) NOT NULL,
#             sources    NVARCHAR(MAX),
#             created_at DATETIME2     DEFAULT GETUTCDATE()
#         );
#     """)
#     cur.execute("CREATE INDEX idx_rag_turns_session ON rag_turns(session_id, created_at);")
#     conn.commit()
#     conn.close()

#     # ── Step 3: Recreate vector index (autocommit — cannot run in transaction) ──
#     print("Step 3/3  Recreating DiskANN vector index...")
#     conn = pyodbc.connect(SQL_CONN_STR, autocommit=True)
#     cur  = conn.cursor()
#     cur.execute("""
#         CREATE VECTOR INDEX idx_rag_embedding
#             ON rag_chunks(embedding)
#             WITH (METRIC = 'COSINE');
#     """)
#     conn.close()

#     print("\n✅ All RAG tables reset. IDENTITY counters reset to 1.")
#     print("   Re-run Cell 7 (ingest) before querying.")


# Uncomment to run:
# reset_all_tables()

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run once. Restart kernel after installation.
# CMD: uv add google-genai pypdf pyodbc rich python-dotenv huggingface_hub fpdf2

# %pip install google-genai pypdf pyodbc rich python-dotenv huggingface_hub fpdf2

In [1]:
# ── Cell 2: Config & API Keys + SQL Server Connection ─────────────────────────

import os, json, pyodbc
from dotenv import load_dotenv

load_dotenv()

# ── API Keys ──────────────────────────────────────────────────────────────────
GEMINI_API_KEY     = os.environ.get("GEMINI_API_KEY", "")
HF_API_KEY         = os.environ.get("HF_API_KEY", "")
EMBED_DIM          = 768

# ── Models ────────────────────────────────────────────────────────────────────
GEMINI_EMBED_MODEL = "gemini-embedding-001"
HF_LLM_MODEL       = "openai/gpt-oss-20b:groq"

# ── SQL Server ────────────────────────────────────────────────────────────────
# Using Windows Authentication — no password needed.
# Update SQL_SERVER to match the server name shown in SSMS title bar.
SQL_SERVER   = "DESKTOP-B6P2IFV"   # ← your server name
SQL_DATABASE = "local_rag"

SQL_CONN_STR = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    f"Trusted_Connection=yes;"
)

# ── Chunking ──────────────────────────────────────────────────────────────────
CHUNK_SIZE         = 800
CHUNK_OVERLAP      = 120
TOP_K              = 5
LLM_MAX_NEW_TOKENS = 1024
LLM_TEMPERATURE    = 0.1

# ── Validate ──────────────────────────────────────────────────────────────────
assert GEMINI_API_KEY, "GEMINI_API_KEY not set in .env"
assert HF_API_KEY,     "HF_API_KEY not set in .env"

def get_conn():
    return pyodbc.connect(SQL_CONN_STR)

# Test connection
try:
    _c = get_conn()
    _cur = _c.cursor()
    _cur.execute("SELECT @@VERSION;")
    _ver = _cur.fetchone()[0].split("\n")[0].strip()
    _cur.execute("SELECT COUNT(*) FROM rag_chunks;")
    _cnt = _cur.fetchone()[0]
    _c.close()
    print(f"SQL Server connected")
    print(f"   {_ver[:70]}")
    print(f"   rag_chunks: {_cnt} rows")
except Exception as e:
    print(f"Connection failed: {e}")

print(f"\nEmbedding : {GEMINI_EMBED_MODEL} | dim={EMBED_DIM}")
print(f"LLM       : {HF_LLM_MODEL}")
print(f"Database  : {SQL_SERVER}/{SQL_DATABASE}")

SQL Server connected
   Microsoft SQL Server 2025 (RTM-GDR) (KB5084814) - 17.0.1110.1 (X64)
   rag_chunks: 0 rows

Embedding : gemini-embedding-001 | dim=768
LLM       : openai/gpt-oss-20b:groq
Database  : DESKTOP-B6P2IFV/local_rag


In [2]:
# ── Cell 3: PDF Text Extraction ───────────────────────────────────────────────

from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> tuple[str, int]:
    reader = PdfReader(pdf_path)
    pages  = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append(f"[Page {i + 1}]\n{text.strip()}")
    return "\n\n".join(pages), len(reader.pages)

print("extract_text_from_pdf() defined")

extract_text_from_pdf() defined


In [3]:
# ── Cell 4: Text Chunking ─────────────────────────────────────────────────────

def chunk_text(text: str) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        end   = start + CHUNK_SIZE
        chunk = text[start:end].strip()
        if len(chunk) >= 80:
            chunks.append(chunk)
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

print("chunk_text() defined")

chunk_text() defined


In [4]:
# ── Cell 5: Gemini Embedding ──────────────────────────────────────────────────

import time
from google import genai
from google.genai import types

genai_client = genai.Client(api_key=GEMINI_API_KEY)


def embed_documents_batch(chunks: list[str]) -> list[list[float]]:
    all_embeddings = []
    total = len(chunks)
    for i, chunk in enumerate(chunks, 1):
        print(f"  Embedding chunk {i}/{total}...", end="\r")
        try:
            result = genai_client.models.embed_content(
                model=GEMINI_EMBED_MODEL,
                contents=chunk,
                config=types.EmbedContentConfig(
                    task_type="RETRIEVAL_DOCUMENT",
                    output_dimensionality=EMBED_DIM,
                ),
            )
            all_embeddings.append(result.embeddings[0].values)
            time.sleep(0.05)
        except Exception as e:
            print(f"\n  Chunk {i} skipped: {e}")
            all_embeddings.append([0.0] * EMBED_DIM)
    print(f"\n  Embedded {len(all_embeddings)} chunks")
    return all_embeddings


def embed_query(text: str) -> list[float]:
    result = genai_client.models.embed_content(
        model=GEMINI_EMBED_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=EMBED_DIM,
        ),
    )
    return result.embeddings[0].values


def vec_to_sql(embedding: list[float]) -> str:
    """SQL Server VECTOR type accepts JSON array strings: '[0.1, 0.2, ...]'"""
    return json.dumps(embedding)


print(f"Embedding functions defined")
print(f"  Model: {GEMINI_EMBED_MODEL} | Dim: {EMBED_DIM}")

Embedding functions defined
  Model: gemini-embedding-001 | Dim: 768


In [5]:
# ── Cell 6: SQL Server Storage & Retrieval ────────────────────────────────────
# Native VECTOR(768) + VECTOR_SEARCH with DiskANN cosine index.

import pyodbc as _pyodbc

def vec_to_json(embedding: list[float]) -> str:
    return json.dumps(embedding)


def get_conn_autocommit():
    """Connection with autocommit=True — required for CREATE/DROP VECTOR INDEX."""
    conn = pyodbc.connect(SQL_CONN_STR, autocommit=True)
    return conn


def drop_vector_index():
    conn = get_conn_autocommit()
    cur  = conn.cursor()
    cur.execute("""
        IF EXISTS (
            SELECT 1 FROM sys.indexes
            WHERE name = 'idx_rag_embedding'
            AND object_id = OBJECT_ID('rag_chunks')
        )
        DROP INDEX idx_rag_embedding ON rag_chunks;
    """)
    conn.close()


def create_vector_index():
    conn = get_conn_autocommit()
    cur  = conn.cursor()
    cur.execute("""
        CREATE VECTOR INDEX idx_rag_embedding
            ON rag_chunks(embedding)
            WITH (METRIC = 'COSINE');
    """)
    conn.close()


def store_in_sqlserver(
    chunks:     list[str],
    embeddings: list[list[float]],
    doc_name:   str,
) -> int:
    """
    SQL Server 2025 preview has two constraints:
    1. Cannot INSERT while VECTOR INDEX exists → drop before insert
    2. CREATE VECTOR INDEX cannot run inside a transaction → autocommit connection
    Pattern: drop (autocommit) → insert (transactional) → recreate (autocommit)
    """
    # Step 1: Drop vector index (autocommit connection)
    print("  Dropping vector index...")
    drop_vector_index()

    # Step 2: Bulk insert (regular transactional connection)
    print(f"  Inserting {len(chunks)} chunks...")
    conn = get_conn()
    cur  = conn.cursor()

    sql = """
        INSERT INTO rag_chunks (source, chunk_index, content, embedding)
        VALUES (?, ?, ?, CAST(? AS VECTOR(768)))
    """
    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        cur.setinputsizes([
            (_pyodbc.SQL_WVARCHAR, 500, 0),
            _pyodbc.SQL_INTEGER,
            (_pyodbc.SQL_WVARCHAR, 0,   0),
            (_pyodbc.SQL_VARCHAR,  0,   0),   # VARCHAR — not NTEXT
        ])
        cur.execute(sql, (doc_name, i, chunk, vec_to_json(emb)))

    conn.commit()
    conn.close()

    # Step 3: Recreate vector index (autocommit connection)
    print("  Recreating vector index...")
    create_vector_index()

    return len(chunks)


print("✅ store_in_sqlserver() defined")
print("   Pattern: drop index → insert → recreate index (autocommit)")


def retrieve_context(query: str) -> list[dict]:
    """
    Retrieves top-K chunks using VECTOR_DISTANCE (exact cosine search).
    VECTOR_DISTANCE is GA in SQL Server 2025 — no index syntax issues.
    For large collections (100k+ chunks) swap back to VECTOR_SEARCH.
    """
    query_embedding = embed_query(query)
    query_vec_json  = vec_to_json(query_embedding)

    conn = get_conn()
    cur  = conn.cursor()

    # Force query vector parameter as VARCHAR (same trick as insert)
    cur.setinputsizes([(_pyodbc.SQL_VARCHAR, 0, 0)])

    cur.execute(f"""
        SELECT TOP ({TOP_K})
            content,
            source,
            chunk_index,
            ingested_at,
            VECTOR_DISTANCE('cosine', embedding, CAST(? AS VECTOR(768))) AS distance
        FROM rag_chunks
        ORDER BY distance ASC;
    """, (query_vec_json,))

    rows = cur.fetchall()
    conn.close()

    return [
        {
            "text":        r[0],
            "source":      r[1],
            "chunk_index": r[2],
            "ingested_at": str(r[3]) if r[3] else "",
            "score":       round(1 - float(r[4]), 4),  # cosine distance → similarity
        }
        for r in rows
    ]


print("✅ retrieve_context() defined — VECTOR_DISTANCE (exact cosine, GA)")

print("Vector store: SQL Server 2025 VECTOR(768) + DiskANN cosine")

✅ store_in_sqlserver() defined
   Pattern: drop index → insert → recreate index (autocommit)
✅ retrieve_context() defined — VECTOR_DISTANCE (exact cosine, GA)
Vector store: SQL Server 2025 VECTOR(768) + DiskANN cosine


In [6]:
# ── Cell 7: RUN INGESTION ─────────────────────────────────────────────────────

PDF_PATH = "./pdfs/attention.pdf"   # CHANGE THIS

assert os.path.exists(PDF_PATH), f"File not found: {PDF_PATH}"
doc_name = os.path.basename(PDF_PATH)
print(f"Ingesting: {doc_name}\n" + "-"*50)

print("Step 1/3  Extracting text...")
raw_text, page_count = extract_text_from_pdf(PDF_PATH)
print(f"  {page_count} pages | {len(raw_text):,} chars")

print("Step 2/3  Chunking...")
chunks = chunk_text(raw_text)
print(f"  {len(chunks)} chunks")

print(f"Step 3/3  Embedding via {GEMINI_EMBED_MODEL}...")
embeddings = embed_documents_batch(chunks)
stored     = store_in_sqlserver(chunks, embeddings, doc_name)

print("-"*50)
print(f"Done! {stored} chunks stored in SQL Server rag_chunks")

Ingesting: attention.pdf
--------------------------------------------------
Step 1/3  Extracting text...
  15 pages | 39,784 chars
Step 2/3  Chunking...
  59 chunks
Step 3/3  Embedding via gemini-embedding-001...
  Embedding chunk 59/59...
  Embedded 59 chunks
  Dropping vector index...
  Inserting 59 chunks...
  Recreating vector index...
--------------------------------------------------
Done! 59 chunks stored in SQL Server rag_chunks


In [7]:
# ── Cell 8: gpt-oss-20b Generation ───────────────────────────────────────────

from huggingface_hub import InferenceClient

hf_client = InferenceClient(api_key=HF_API_KEY)

SYSTEM_MSG = (
    "You are a precise, document-grounded assistant. "
    "Answer ONLY using the CONTEXT provided. "
    "If the context is insufficient, say: "
    "'The provided documents do not cover this sufficiently.' "
    "Never fabricate facts. Be concise. Use bullet points for multi-part answers."
)

def build_messages(query: str, context_chunks: list[dict]) -> list[dict]:
    context_str = "\n\n---\n\n".join([
        f"[Source: {c['source']} | Chunk #{c['chunk_index']} | Relevance: {c['score']}]\n{c['text']}"
        for c in context_chunks
    ])
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"CONTEXT:\n{context_str}\n\nQUESTION:\n{query}"},
    ]

def generate_answer(messages: list[dict]) -> str:
    completion = hf_client.chat.completions.create(
        model=HF_LLM_MODEL,
        messages=messages,
        max_tokens=LLM_MAX_NEW_TOKENS,
        temperature=LLM_TEMPERATURE,
    )
    return completion.choices[0].message.content.strip()

print(f"gpt-oss-20b generation ready: {HF_LLM_MODEL}")

gpt-oss-20b generation ready: openai/gpt-oss-20b:groq


In [8]:
# ── Cell 9: Full RAG Pipeline — ask() ─────────────────────────────────────────

def ask(query: str, show_chunks: bool = True):
    print(f"\n{'='*60}\n Question: {query}\n{'='*60}")
    print("Retrieving from SQL Server...")
    chunks = retrieve_context(query)

    if not chunks:
        print("No chunks found. Run Cell 7 first."); return

    if show_chunks:
        print(f"\nTop {len(chunks)} chunks:")
        for i, c in enumerate(chunks, 1):
            bar = "█"*int(c["score"]*30) + "░"*(30-int(c["score"]*30))
            print(f"  {i}. {c['source']} #{c['chunk_index']}  [{bar}]  {c['score']}")

    print(f"\nGenerating via {HF_LLM_MODEL}...\n")
    try:
        answer = generate_answer(build_messages(query, chunks))
        print(f"{'─'*60}\n Answer:\n{'─'*60}\n{answer}\n{'─'*60}")
        return answer
    except Exception as e:
        if "503" in str(e) or "loading" in str(e).lower():
            print("Model cold-starting. Wait 30s and retry.")
        elif "429" in str(e):
            print("Rate limit. Wait 60s and retry.")
        else:
            print(f"LLM error: {e}")

print("ask() defined")

ask() defined


In [9]:
# ── Cell 10: ASK A SINGLE QUESTION ───────────────────────────────────────────

ask("What is the main topic of this document?")


 Question: What is the main topic of this document?
Retrieving from SQL Server...

Top 5 chunks:
  1. attention.pdf #0  [███████████████████░░░░░░░░░░░]  0.6529
  2. attention.pdf #56  [███████████████████░░░░░░░░░░░]  0.6498
  3. attention.pdf #58  [███████████████████░░░░░░░░░░░]  0.6476
  4. attention.pdf #57  [███████████████████░░░░░░░░░░░]  0.6474
  5. attention.pdf #54  [███████████████████░░░░░░░░░░░]  0.6399

Generating via openai/gpt-oss-20b:groq...

────────────────────────────────────────────────────────────
 Answer:
────────────────────────────────────────────────────────────
- The document presents the **Transformer** architecture, a sequence‑to‑sequence model that relies entirely on self‑attention mechanisms for tasks such as machine translation.
────────────────────────────────────────────────────────────


'- The document presents the **Transformer** architecture, a sequence‑to‑sequence model that relies entirely on self‑attention mechanisms for tasks such as machine translation.'

In [10]:
# ── Cell 11: INTERACTIVE REPL ─────────────────────────────────────────────────

print(f"RAG REPL | LLM: {HF_LLM_MODEL} | DB: SQL Server 2025")
print("Type 'exit' to quit.\n")

while True:
    try:
        q = input("You: ").strip()
        if not q: continue
        if q.lower() in ("exit","quit","q"): print("Goodbye!"); break
        ask(q)
    except KeyboardInterrupt:
        print("\nInterrupted."); break

RAG REPL | LLM: openai/gpt-oss-20b:groq | DB: SQL Server 2025
Type 'exit' to quit.

Goodbye!


In [11]:
# ── Cell 12: UTILITIES ────────────────────────────────────────────────────────

def check_db_stats():
    conn = get_conn(); cur = conn.cursor()
    print(f"\nSQL Server RAG table stats")
    for t in ["rag_chunks","rag_staleness","rag_chunk_registry","rag_sessions","rag_turns"]:
        cur.execute(f"SELECT COUNT(*) FROM {t};")
        print(f"  {t:<25} {cur.fetchone()[0]:>8} rows")
    conn.close()

def list_ingested_docs():
    conn = get_conn(); cur = conn.cursor()
    cur.execute("""
        SELECT source, COUNT(*) AS chunks, MIN(ingested_at)
        FROM rag_chunks GROUP BY source ORDER BY MIN(ingested_at);
    """)
    for r in cur.fetchall():
        print(f"  {r[0]}  ({r[1]} chunks, {str(r[2])[:19]})")
    conn.close()

def wipe_db():
    if input("Type 'yes' to wipe all RAG tables: ").strip().lower() != "yes":
        print("Cancelled."); return
    conn = get_conn(); cur = conn.cursor()
    for t in ["rag_turns","rag_sessions","rag_chunk_registry","rag_staleness","rag_chunks"]:
        cur.execute(f"DELETE FROM {t};")
    conn.commit(); conn.close()
    print("All RAG tables wiped.")

check_db_stats()
# list_ingested_docs()
# wipe_db()


SQL Server RAG table stats
  rag_chunks                      59 rows
  rag_staleness                    0 rows
  rag_chunk_registry               0 rows
  rag_sessions                     0 rows
  rag_turns                        0 rows


In [12]:
# ── Cell 13: SQL Server Direct Query Explorer ──────────────────────────────────

def peek(n=5):
    conn=get_conn(); cur=conn.cursor()
    cur.execute("SELECT TOP(?) source,chunk_index,content,ingested_at FROM rag_chunks ORDER BY ingested_at,chunk_index;",(n,))
    for r in cur.fetchall():
        print(f"[#{r[1]}] {r[0]} | {str(r[3])[:19]}\n  {r[2][:200]}...\n")
    conn.close()

def get_chunks_by_source(src):
    conn=get_conn(); cur=conn.cursor()
    cur.execute("SELECT chunk_index,content FROM rag_chunks WHERE source=? ORDER BY chunk_index;",(src,))
    for r in cur.fetchall(): print(f"  #{r[0]:>3} | {r[1][:150]}...")
    conn.close()

def get_chunk(src, idx):
    conn=get_conn(); cur=conn.cursor()
    cur.execute("SELECT content,ingested_at FROM rag_chunks WHERE source=? AND chunk_index=?;",(src,idx))
    r=cur.fetchone(); conn.close()
    if r: print(f"{src} #{idx} | {str(r[1])[:19]}\n{r[0]}")
    else: print("Not found.")

def semantic_search(query):
    for c in retrieve_context(query):
        bar="█"*int(c['score']*30)+"░"*(30-int(c['score']*30))
        print(f"  [{bar}] {c['score']} | {c['source']} #{c['chunk_index']}\n  {c['text'][:250]}...\n")

def keyword_search(kw):
    conn=get_conn(); cur=conn.cursor()
    cur.execute("SELECT source,chunk_index,content FROM rag_chunks WHERE content LIKE ? ORDER BY source,chunk_index;",(f"%{kw}%",))
    for r in cur.fetchall():
        idx=r[2].lower().find(kw.lower())
        print(f"  {r[0]} #{r[1]}:\n  ...{r[2][max(0,idx-80):idx+120]}...\n")
    conn.close()

peek(n=3)
# get_chunks_by_source("attention.pdf")
# get_chunk("attention.pdf", 6)
# semantic_search("multi-head attention")
# keyword_search("BLEU")

[#0] attention.pdf | 2026-04-27 06:47:13
  [Page 1]
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All ...

[#1] attention.pdf | 2026-04-27 06:47:13
  ent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple netw...

[#2] attention.pdf | 2026-04-27 06:47:13
  -to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best ...



In [13]:
# ── Cell 13A: Conversation Memory (SQL Server backed) ─────────────────────────

from datetime import datetime

MAX_HISTORY_TURNS = 6


class ConversationMemory:
    def __init__(self, session_id: str = None):
        self.session_id  = session_id or datetime.now().strftime("%Y%m%d_%H%M%S")
        self._turn_count = 0
        conn = get_conn(); cur = conn.cursor()
        cur.execute("SELECT turn_count FROM rag_sessions WHERE session_id=?;",(self.session_id,))
        row = cur.fetchone()
        if row:
            self._turn_count = row[0]
            conn.close()
            print(f"Resumed: {self.session_id} ({self._turn_count} turns)")
        else:
            cur.execute("INSERT INTO rag_sessions(session_id,model,embed_model) VALUES(?,?,?);",(self.session_id,HF_LLM_MODEL,GEMINI_EMBED_MODEL))
            conn.commit(); conn.close()
            print(f"New session: {self.session_id}")

    @property
    def metadata(self): return {"turns":self._turn_count,"session_id":self.session_id}

    def add_turn(self, question, answer, chunks):
        sources = json.dumps([f"{c['source']} #{c['chunk_index']} ({c['score']})" for c in chunks])
        conn=get_conn(); cur=conn.cursor()
        cur.execute("INSERT INTO rag_turns(session_id,role,content) VALUES(?,'user',?);",(self.session_id,question))
        cur.execute("INSERT INTO rag_turns(session_id,role,content,sources) VALUES(?,'assistant',?,?);",(self.session_id,answer,sources))
        self._turn_count+=1
        cur.execute("UPDATE rag_sessions SET turn_count=?,updated_at=GETUTCDATE() WHERE session_id=?;",(self._turn_count,self.session_id))
        conn.commit(); conn.close()

    def get_messages_with_history(self, query, context_chunks, system_msg):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("SELECT TOP(?) role,content FROM rag_turns WHERE session_id=? ORDER BY created_at DESC;",(MAX_HISTORY_TURNS*2,self.session_id))
        history=list(reversed(cur.fetchall())); conn.close()
        msgs=[{"role":"system","content":system_msg}]
        for role,content in history: msgs.append({"role":role,"content":content})
        ctx="\n\n---\n\n".join([f"[{c['source']} #{c['chunk_index']} {c['score']}]\n{c['text']}" for c in context_chunks])
        msgs.append({"role":"user","content":f"CONTEXT:\n{ctx}\n\nQUESTION:\n{query}"})
        return msgs

    def show_history(self):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("SELECT role,content,sources,created_at FROM rag_turns WHERE session_id=? ORDER BY created_at;",(self.session_id,))
        for role,content,sources,ts in cur.fetchall():
            icon="🧑" if role=="user" else "🤖"
            print(f"{icon} [{str(ts)[:19]}]:\n  {content[:300]}")
        conn.close()

    def clear(self):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("DELETE FROM rag_turns WHERE session_id=?;",(self.session_id,))
        cur.execute("UPDATE rag_sessions SET turn_count=0 WHERE session_id=?;",(self.session_id,))
        conn.commit(); conn.close(); self._turn_count=0
        print(f"Cleared session: {self.session_id}")

    def summary(self):
        print(f"Session: {self.session_id} | Turns: {self._turn_count} | Storage: SQL Server")


def list_sessions():
    conn=get_conn(); cur=conn.cursor()
    cur.execute("SELECT session_id,turn_count,created_at,updated_at FROM rag_sessions ORDER BY created_at DESC;")
    for r in cur.fetchall():
        print(f"  {r[0]} | turns:{r[1]} | created:{str(r[2])[:19]}")
    conn.close()


print(f"ConversationMemory (SQL Server) defined | Max history: {MAX_HISTORY_TURNS} turns")

ConversationMemory (SQL Server) defined | Max history: 6 turns


In [14]:
# ── Cell 13B: Memory-aware ask_with_memory() ───────────────────────────────────

SYSTEM_MSG_MEMORY = (
    "You are a precise, document-grounded assistant with memory of our conversation. "
    "Answer using ONLY the CONTEXT and prior turns. Never fabricate facts."
)


def ask_with_memory(query: str, memory: ConversationMemory, show_chunks: bool = True):
    print(f"\n{'='*60}\n Question: {query}\n Session: {memory.session_id} | Turn #{memory.metadata['turns']+1}\n{'='*60}")
    chunks = retrieve_context(query)
    if not chunks: print("No chunks. Run ingestion first."); return
    if show_chunks:
        for i,c in enumerate(chunks,1):
            bar="█"*int(c['score']*30)+"░"*(30-int(c['score']*30))
            print(f"  {i}. {c['source']} #{c['chunk_index']} [{bar}] {c['score']}")
    try:
        messages = memory.get_messages_with_history(query, chunks, SYSTEM_MSG_MEMORY)
        answer   = generate_answer(messages)
        print(f"\n{'─'*60}\n{answer}\n{'─'*60}")
        memory.add_turn(query, answer, chunks)
        print(f"Checkpointed to SQL Server (turn #{memory.metadata['turns']})")
        return answer
    except Exception as e:
        print(f"LLM error: {e}")


print("ask_with_memory() defined")

ask_with_memory() defined


In [15]:
# ── Cell 13C: Interactive Memory REPL ─────────────────────────────────────────

memory = ConversationMemory()
# memory = ConversationMemory("paste_session_id")  # resume
memory.summary()
print("Commands: history | summary | clear | sessions | exit\n")

while True:
    try:
        q = input("You: ").strip()
        if not q: continue
        if q.lower() in ("exit","quit"): print("Goodbye!"); break
        elif q.lower() == "history": memory.show_history()
        elif q.lower() == "summary": memory.summary()
        elif q.lower() == "clear":   memory.clear()
        elif q.lower() == "sessions": list_sessions()
        else: ask_with_memory(q, memory)
    except KeyboardInterrupt:
        print("\nInterrupted."); break

New session: 20260427_121744
Session: 20260427_121744 | Turns: 0 | Storage: SQL Server
Commands: history | summary | clear | sessions | exit

Goodbye!


In [16]:
# ── Cell 13D: Initialize Memory Session ───────────────────────────────────────

memory = ConversationMemory()
memory.summary()

New session: 20260427_121750
Session: 20260427_121750 | Turns: 0 | Storage: SQL Server


In [17]:
# ── Cell 13E: Ask Single Question With Memory ─────────────────────────────────

ask_with_memory("What is the main topic of this document?", memory)


 Question: What is the main topic of this document?
 Session: 20260427_121750 | Turn #1
  1. attention.pdf #0 [███████████████████░░░░░░░░░░░] 0.6529
  2. attention.pdf #56 [███████████████████░░░░░░░░░░░] 0.6498
  3. attention.pdf #58 [███████████████████░░░░░░░░░░░] 0.6476
  4. attention.pdf #57 [███████████████████░░░░░░░░░░░] 0.6474
  5. attention.pdf #54 [███████████████████░░░░░░░░░░░] 0.6399

────────────────────────────────────────────────────────────
The document is the research paper **“Attention Is All You Need.”** It introduces and discusses the Transformer architecture, a new sequence‑to‑sequence model that relies entirely on self‑attention mechanisms (without recurrent or convolutional layers) for tasks such as machine translation.
────────────────────────────────────────────────────────────
Checkpointed to SQL Server (turn #1)


'The document is the research paper **“Attention Is All You Need.”** It introduces and discusses the Transformer architecture, a new sequence‑to‑sequence model that relies entirely on self‑attention mechanisms (without recurrent or convolutional layers) for tasks such as machine translation.'

In [18]:
# ── Cell 13F: Ask Multiple Questions With Memory ──────────────────────────────

for q in [
    "What architecture does this paper propose?",
    "How many attention heads does it use?",
    "What optimizer was used for training?",
    "How does that compare to what you said about the architecture?",
]:
    ask_with_memory(q, memory)


 Question: What architecture does this paper propose?
 Session: 20260427_121750 | Turn #2
  1. attention.pdf #6 [█████████████████████░░░░░░░░░] 0.7242
  2. attention.pdf #9 [█████████████████████░░░░░░░░░] 0.7182
  3. attention.pdf #10 [█████████████████████░░░░░░░░░] 0.7162
  4. attention.pdf #1 [█████████████████████░░░░░░░░░] 0.7152
  5. attention.pdf #11 [█████████████████████░░░░░░░░░] 0.7079

────────────────────────────────────────────────────────────
The paper proposes the **Transformer** architecture—a pure‑attention, encoder‑decoder model that replaces recurrent or convolutional layers with stacked multi‑head self‑attention and position‑wise feed‑forward sub‑layers.
────────────────────────────────────────────────────────────
Checkpointed to SQL Server (turn #2)

 Question: How many attention heads does it use?
 Session: 20260427_121750 | Turn #3
  1. attention.pdf #17 [███████████████████████░░░░░░░] 0.7962
  2. attention.pdf #16 [███████████████████████░░░░░░░] 0.7669
  3

In [19]:
# ── Cell 14A: Staleness Tracking (SQL Server) ─────────────────────────────────

import hashlib

def compute_file_hash(pdf_path):
    sha=hashlib.sha256()
    with open(pdf_path,"rb") as f:
        for block in iter(lambda: f.read(65536), b""): sha.update(block)
    return sha.hexdigest()

def compute_chunk_hash(text): return hashlib.md5(text.encode()).hexdigest()


class StalenessRegistry:
    def register(self, doc_name, file_hash, chunk_count):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("""
            MERGE rag_staleness AS t
            USING (SELECT ? doc_name,? file_hash,? chunk_count) AS s ON t.doc_name=s.doc_name
            WHEN MATCHED THEN UPDATE SET file_hash=s.file_hash,chunk_count=s.chunk_count,ingested_at=GETUTCDATE(),version=t.version+1
            WHEN NOT MATCHED THEN INSERT(doc_name,file_hash,chunk_count,version) VALUES(s.doc_name,s.file_hash,s.chunk_count,1);
        """,(doc_name,file_hash,chunk_count))
        conn.commit(); conn.close()

    def check(self, pdf_path):
        doc_name=os.path.basename(pdf_path); h=compute_file_hash(pdf_path)
        conn=get_conn(); cur=conn.cursor()
        cur.execute("SELECT file_hash,ingested_at,version FROM rag_staleness WHERE doc_name=?;",(doc_name,))
        row=cur.fetchone(); conn.close()
        if not row: return {"status":"new","doc_name":doc_name,"file_hash":h}
        if row[0]==h: return {"status":"unchanged","doc_name":doc_name,"file_hash":h,"ingested_at":str(row[1]),"version":row[2]}
        return {"status":"stale","doc_name":doc_name,"file_hash":h,"old_hash":row[0],"ingested_at":str(row[1]),"version":row[2]}

    def show(self):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("SELECT doc_name,version,ingested_at,chunk_count,file_hash FROM rag_staleness;")
        for r in cur.fetchall():
            print(f"  {r[0]} | v{r[1]} | {str(r[2])[:19]} | {r[3]} chunks | {r[4][:20]}...")
        conn.close()


class ChunkRegistry:
    def get_doc(self, doc_name):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("SELECT chunk_hash,chunk_id FROM rag_chunk_registry WHERE doc_name=?;",(doc_name,))
        r={r[0]:r[1] for r in cur.fetchall()}; conn.close(); return r

    def update(self, doc_name, registry):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("DELETE FROM rag_chunk_registry WHERE doc_name=?;",(doc_name,))
        for h,cid in registry.items():
            cur.execute("INSERT INTO rag_chunk_registry(doc_name,chunk_hash,chunk_id) VALUES(?,?,?);",(doc_name,h,str(cid)))
        conn.commit(); conn.close()

    def show(self):
        conn=get_conn(); cur=conn.cursor()
        cur.execute("SELECT doc_name,COUNT(*) FROM rag_chunk_registry GROUP BY doc_name;")
        for r in cur.fetchall(): print(f"  {r[0]}: {r[1]} chunks")
        conn.close()


def store_in_sqlserver_v2(
    chunks: list[str], embeddings: list[list[float]],
    doc_name: str, file_hash: str, ingested_at: str
) -> dict:
    # Drop index first (autocommit)
    drop_vector_index()

    conn = get_conn()
    cur  = conn.cursor()
    hash_to_id = {}

    sql_v2 = """
        INSERT INTO rag_chunks
            (source, chunk_index, content, file_hash, ingested_at, embedding)
        OUTPUT INSERTED.id
        VALUES (?, ?, ?, ?, ?, CAST(? AS VECTOR(768)));
    """
    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        cur.setinputsizes([
            (_pyodbc.SQL_WVARCHAR, 500, 0),
            _pyodbc.SQL_INTEGER,
            (_pyodbc.SQL_WVARCHAR, 0,   0),
            (_pyodbc.SQL_WVARCHAR, 64,  0),
            (_pyodbc.SQL_WVARCHAR, 50,  0),
            (_pyodbc.SQL_VARCHAR,  0,   0),
        ])
        cur.execute(sql_v2, (doc_name, i, chunk, file_hash, ingested_at, vec_to_json(emb)))
        hash_to_id[compute_chunk_hash(chunk)] = cur.fetchone()[0]

    conn.commit()
    conn.close()

    # Recreate index (autocommit)
    create_vector_index()
    return hash_to_id


staleness_reg=StalenessRegistry()
chunk_reg=ChunkRegistry()
print("Cell 14A: Staleness + Chunk registries ready")
staleness_reg.show()
chunk_reg.show()

Cell 14A: Staleness + Chunk registries ready


In [20]:
# ── Cell 14B: CDC Engine + Smart Ingest (SQL Server) ──────────────────────────
# KEY FIX: drop_vector_index() is called BEFORE both DELETE and INSERT.
# SQL Server 2025 preview does not allow any data modification (DELETE, INSERT,
# UPDATE) on a table that has a vector index on it.
# Pattern: drop index → DELETE removed → INSERT added → recreate index

from datetime import datetime as _dt

def diff_chunks(old_registry: dict, new_chunks: list[str]) -> dict:
    new_hash_map = {compute_chunk_hash(c): c for c in new_chunks}
    old_hashes   = set(old_registry.keys())
    new_hashes   = set(new_hash_map.keys())
    return {
        "added":     {h: new_hash_map[h] for h in (new_hashes - old_hashes)},
        "removed":   {h: old_registry[h] for h in (old_hashes - new_hashes)},
        "unchanged": {h: old_registry[h] for h in (old_hashes & new_hashes)},
    }


def cdc_update(pdf_path: str) -> dict:
    """
    Surgical CDC update for a changed PDF.
    SQL Server 2025 preview: NO data modification allowed while vector
    index exists — drop index FIRST, do all changes, recreate index LAST.

    Order:
      1. Extract + chunk new version
      2. Diff old vs new chunk hashes
      3. Drop vector index          ← must happen before ANY modification
      4. Delete removed chunks
      5. Embed + insert added chunks
      6. Recreate vector index      ← once at the end
      7. Update staleness + chunk registries
    """
    doc_name    = os.path.basename(pdf_path)
    file_hash   = compute_file_hash(pdf_path)
    ingested_at = _dt.utcnow().isoformat()

    # ── Step 1: Extract + Chunk ───────────────────────────────────────────────
    print("\n📄 Extracting new version...")
    raw_text, page_count = extract_text_from_pdf(pdf_path)
    new_chunks = chunk_text(raw_text)
    print(f"   {page_count} pages → {len(new_chunks)} chunks")

    # ── Step 2: Diff ──────────────────────────────────────────────────────────
    old_registry = chunk_reg.get_doc(doc_name)
    diff         = diff_chunks(old_registry, new_chunks)

    n_added     = len(diff["added"])
    n_removed   = len(diff["removed"])
    n_unchanged = len(diff["unchanged"])
    total       = n_unchanged + n_added

    print(f"\n📊 CDC diff result:")
    print(f"   ✅ Unchanged : {n_unchanged:>4}  (kept — zero re-embedding cost)")
    print(f"   ➕ Added     : {n_added:>4}  (will be embedded + inserted)")
    print(f"   ➖ Removed   : {n_removed:>4}  (will be deleted from rag_chunks)")
    if total > 0:
        print(f"   💰 API calls saved: {n_unchanged}/{total} ({int(100*n_unchanged/total)}% reuse)")

    new_doc_registry = dict(diff["unchanged"])

    # ── Step 3: Drop vector index FIRST ───────────────────────────────────────
    # Must happen before ANY modification (DELETE or INSERT).
    # Uses autocommit connection — cannot run inside a transaction.
    if n_removed > 0 or n_added > 0:
        print("  Dropping vector index (required before any modification)...")
        drop_vector_index()

    # ── Step 4: Delete removed chunks ─────────────────────────────────────────
    # Safe now — vector index has been dropped.
    conn = get_conn()
    cur  = conn.cursor()

    if diff["removed"]:
        ids_to_delete = [int(v) for v in diff["removed"].values()]
        placeholders  = ",".join(["?"] * len(ids_to_delete))
        cur.execute(
            f"DELETE FROM rag_chunks WHERE id IN ({placeholders});",
            ids_to_delete
        )
        conn.commit()
        print(f"   🗑️  Deleted {len(ids_to_delete)} stale chunks")

    # ── Step 5: Embed + Insert added chunks ───────────────────────────────────
    if diff["added"]:
        added_texts  = list(diff["added"].values())
        added_hashes = list(diff["added"].keys())

        print(f"\n🧠 Embedding {len(added_texts)} new/changed chunks...")
        new_embeddings = embed_documents_batch(added_texts)

        # Get next chunk_index for this document
        cur.execute(
            "SELECT ISNULL(MAX(chunk_index), -1) FROM rag_chunks WHERE source = ?;",
            (doc_name,)
        )
        next_idx = cur.fetchone()[0] + 1

        cdc_sql = """
            INSERT INTO rag_chunks
                (source, chunk_index, content, file_hash, ingested_at, embedding)
            OUTPUT INSERTED.id
            VALUES (?, ?, ?, ?, ?, CAST(? AS VECTOR(768)));
        """
        for i, (h, text, emb) in enumerate(
            zip(added_hashes, added_texts, new_embeddings)
        ):
            cur.setinputsizes([
                (_pyodbc.SQL_WVARCHAR, 500, 0),   # source
                _pyodbc.SQL_INTEGER,               # chunk_index
                (_pyodbc.SQL_WVARCHAR, 0,   0),   # content
                (_pyodbc.SQL_WVARCHAR, 64,  0),   # file_hash
                (_pyodbc.SQL_WVARCHAR, 50,  0),   # ingested_at
                (_pyodbc.SQL_VARCHAR,  0,   0),   # embedding ← VARCHAR not NTEXT
            ])
            cur.execute(
                cdc_sql,
                (doc_name, next_idx + i, text, file_hash, ingested_at, vec_to_json(emb))
            )
            new_doc_registry[h] = cur.fetchone()[0]

        conn.commit()
        print(f"   ✅ Inserted {len(added_texts)} new chunks")

    conn.close()

    # ── Step 6: Recreate vector index ONCE at the end ─────────────────────────
    # Uses autocommit connection — cannot run inside a transaction.
    if n_removed > 0 or n_added > 0:
        print("  Recreating vector index...")
        create_vector_index()

    # ── Step 7: Update registries ─────────────────────────────────────────────
    chunk_reg.update(doc_name, new_doc_registry)
    staleness_reg.register(doc_name, file_hash, len(new_doc_registry))
    print(f"\n💾 CDC complete — rag_staleness + rag_chunk_registry updated")

    return {
        "added":     n_added,
        "removed":   n_removed,
        "unchanged": n_unchanged,
    }


def ingest_pdf_v2(pdf_path: str, force: bool = False):
    """Smart ingest: new → full ingest | stale → CDC | unchanged → skip."""
    doc_name = os.path.basename(pdf_path)
    print(f"\n{'='*60}\n📥 Smart Ingest: {doc_name}\n{'='*60}")

    status = staleness_reg.check(pdf_path)
    print(f"📌 Status: {status['status'].upper()}")

    if status["status"] == "unchanged" and not force:
        print(f"✅ Already up to date (v{status['version']}). Use force=True to re-ingest.")
        return

    if status["status"] == "stale":
        print(f"🔄 Changes detected (was v{status['version']}) — running CDC...")
        return cdc_update(pdf_path)

    # ── Full ingest (new or forced) ───────────────────────────────────────────
    ingested_at = _dt.utcnow().isoformat()
    file_hash   = status["file_hash"]

    print("📄 Step 1/3  Extracting...")
    raw_text, page_count = extract_text_from_pdf(pdf_path)
    print(f"   ✅ {page_count} pages | {len(raw_text):,} chars")

    print("✂️  Step 2/3  Chunking...")
    chunks = chunk_text(raw_text)
    print(f"   ✅ {len(chunks)} chunks")

    print(f"🧠 Step 3/3  Embedding via {GEMINI_EMBED_MODEL}...")
    embeddings     = embed_documents_batch(chunks)
    chunk_hash_map = store_in_sqlserver_v2(
        chunks, embeddings, doc_name, file_hash, ingested_at
    )
    chunk_reg.update(doc_name, chunk_hash_map)
    staleness_reg.register(doc_name, file_hash, len(chunks))

    print(f"\n✅ Full ingest complete! {len(chunks)} chunks in rag_chunks")
    print(f"   Hash: {file_hash[:24]}... | Ingested: {ingested_at[:19]}")


print("✅ Cell 14B: CDC Engine + Smart Ingest ready")
print("   Fix: drop_vector_index() now runs before DELETE and INSERT")


✅ Cell 14B: CDC Engine + Smart Ingest ready
   Fix: drop_vector_index() now runs before DELETE and INSERT


In [21]:
# ── Cell 14C: Recency-Weighted Retrieval ──────────────────────────────────────

import math
from datetime import datetime, timezone

RECENCY_ALPHA=0.85
RECENCY_HALF_LIFE=30


def recency_decay(ingested_at_str, half_life=RECENCY_HALF_LIFE):
    try:
        dt=datetime.fromisoformat(str(ingested_at_str).replace("Z","+00:00"))
        if dt.tzinfo is None: dt=dt.replace(tzinfo=timezone.utc)
        days=(datetime.now(timezone.utc)-dt).total_seconds()/86400
        return round(math.exp(-math.log(2)/half_life*days),4)
    except: return 0.5


def retrieve_context_weighted(
    query: str, alpha: float = RECENCY_ALPHA, show_scores: bool = False
) -> list[dict]:
    query_embedding = embed_query(query)
    query_vec_json  = vec_to_json(query_embedding)
    n_candidates    = TOP_K * 3

    conn = get_conn()
    cur  = conn.cursor()

    cur.setinputsizes([(_pyodbc.SQL_VARCHAR, 0, 0)])

    cur.execute(f"""
        SELECT TOP ({n_candidates})
            content, source, chunk_index, ingested_at,
            VECTOR_DISTANCE('cosine', embedding, CAST(? AS VECTOR(768))) AS distance
        FROM rag_chunks
        ORDER BY distance ASC;
    """, (query_vec_json,))

    rows = cur.fetchall()
    conn.close()

    chunks = []
    for r in rows:
        cosine_score  = round(1 - float(r[4]), 4)
        ingested_at   = str(r[3]) if r[3] else _dt2.now(timezone.utc).isoformat()
        recency_score = recency_decay(ingested_at)
        blended       = round(alpha * cosine_score + (1 - alpha) * recency_score, 4)
        chunks.append({
            "text":          r[0], "source": r[1], "chunk_index": r[2],
            "ingested_at":   str(r[3])[:19] if r[3] else "",
            "cosine_score":  cosine_score,
            "recency_score": recency_score,
            "score":         blended,
        })

    chunks.sort(key=lambda x: x["score"], reverse=True)
    top_chunks = chunks[:TOP_K]

    if show_scores:
        print(f"\n📊 Score breakdown (α={alpha} cosine | {round(1-alpha,2)} recency):")
        print(f"   {'#':>3}  {'chunk':>7}  {'cosine':>8}  {'recency':>8}  {'blended':>8}")
        print(f"   {'─'*50}")
        for i, c in enumerate(top_chunks, 1):
            print(f"   {i:>3}  #{c['chunk_index']:>5}  "
                  f"{c['cosine_score']:>8}  {c['recency_score']:>8}  {c['score']:>8}")

    return top_chunks


def ask_weighted(query, memory=None, alpha=RECENCY_ALPHA, show_chunks=True, show_scores=False):
    print(f"\n{'='*60}\n {query}\n alpha={alpha}\n{'='*60}")
    chunks=retrieve_context_weighted(query,alpha,show_scores)
    if not chunks: print("No chunks. Run ingest_pdf_v2() first."); return
    if show_chunks:
        for i,c in enumerate(chunks,1):
            bar="█"*int(c['score']*30)+"░"*(30-int(c['score']*30))
            print(f"  {i}. {c['source']} #{c['chunk_index']} [{bar}] {c['score']} cos={c['cosine_score']} rec={c['recency_score']}")
    msgs=(memory.get_messages_with_history(query,chunks,SYSTEM_MSG_MEMORY)
          if memory else build_messages(query,chunks))
    try:
        answer=generate_answer(msgs)
        print(f"\n{'─'*60}\n{answer}\n{'─'*60}")
        if memory:
            memory.add_turn(query,answer,chunks)
            print(f"Checkpointed (turn #{memory.metadata['turns']})")
        return answer
    except Exception as e: print(f"LLM error: {e}")


print(f"Cell 14C: Recency retrieval ready | alpha={RECENCY_ALPHA} | half-life={RECENCY_HALF_LIFE}d")

Cell 14C: Recency retrieval ready | alpha=0.85 | half-life=30d


In [22]:
# ── Cell 14D: Test PDF Generator ──────────────────────────────────────────────

from fpdf import FPDF

os.makedirs("./pdfs/versions", exist_ok=True)

def make_pdf(path, title, sections):
    def s(t):
        for c,r in {"\u2014":"-","\u2013":"-","\u2018":"'","\u2019":"'","\u201c":'"',"\u201d":'"',"\u2026":"...","\u00a0":" "}.items(): t=t.replace(c,r)
        return t.encode("latin-1",errors="ignore").decode("latin-1")
    pdf=FPDF(); pdf.set_margins(20,20,20); pdf.add_page()
    pdf.set_font("Helvetica","B",18); pdf.cell(0,14,s(title),new_x="LMARGIN",new_y="NEXT"); pdf.ln(4)
    for h,b in sections.items():
        pdf.set_font("Helvetica","B",13); pdf.cell(0,10,s(h),new_x="LMARGIN",new_y="NEXT")
        pdf.set_font("Helvetica",size=11); pdf.multi_cell(0,7,s(b)); pdf.ln(5)
    pdf.output(path); print(f"Created: {path}")

INTRO="Retrieval-Augmented Generation (RAG) combines retrieval with an LLM to ground answers in verifiable source material and reduce hallucinations."
EMBED="Embedding models map text to dense vectors. Google gemini-embedding-001 leads MTEB leaderboard. Asymmetric retrieval (RETRIEVAL_DOCUMENT vs RETRIEVAL_QUERY) is critical for RAG quality."
VDB1="Vector databases store embeddings for ANN search. Options: ChromaDB, FAISS, Pinecone. HNSW indexing provides sub-linear query time."
VDB2="Vector databases store embeddings for ANN search. SQL Server 2025 now provides native VECTOR type with DiskANN indexing. Other options: pgvector, Qdrant, Weaviate."
LLM2="LLM generation phase uses retrieved context. Use temperature 0.1 for factual RAG. Models: GPT-4o, gpt-oss-20b, Llama 3.1."
LLM3="LLM generation in 2025: gpt-oss-20b via Groq offers excellent free-tier performance. Re-ranking cross-encoders now standard in production."
EVAL="RAG evaluation: MRR, NDCG, Recall@K for retrieval. Faithfulness, Answer Relevance for generation. RAGAS automates LLM-as-judge evaluation."

ACTIVE="./pdfs/versions/rag_guide.pdf"
V2="./pdfs/versions/rag_guide_v2_staged.pdf"
V3="./pdfs/versions/rag_guide_v3_staged.pdf"

make_pdf(ACTIVE,"RAG Guide - v1",{"1. Introduction":INTRO,"2. Vector Databases":VDB1,"3. Embedding Models":EMBED})
make_pdf(V2,"RAG Guide - v2",{"1. Introduction":INTRO,"2. Vector Databases":VDB2,"3. Embedding Models":EMBED,"4. LLM Integration":LLM2})
make_pdf(V3,"RAG Guide - v3",{"1. Introduction":INTRO,"2. Vector Databases":VDB2,"3. Embedding Models":EMBED,"4. LLM Integration":LLM3,"5. RAG Evaluation":EVAL})

print("3 test PDFs ready in ./pdfs/versions/")

Created: ./pdfs/versions/rag_guide.pdf
Created: ./pdfs/versions/rag_guide_v2_staged.pdf
Created: ./pdfs/versions/rag_guide_v3_staged.pdf
3 test PDFs ready in ./pdfs/versions/


In [23]:
# ── Cell 14E: Full Test Suite ─────────────────────────────────────────────────

import time, shutil

ACTIVE="./pdfs/versions/rag_guide.pdf"
V2="./pdfs/versions/rag_guide_v2_staged.pdf"
V3="./pdfs/versions/rag_guide_v3_staged.pdf"

print("TEST 1 — Full ingest v1")
ingest_pdf_v2(ACTIVE)
staleness_reg.show()

print("\nTEST 2 — Staleness check (unchanged)")
ingest_pdf_v2(ACTIVE)

print("\nTEST 3 — Queries against v1")
ask_weighted("What vector databases does this guide mention?", show_scores=True)

print("\nTEST 4 — Upgrade to v2 (CDC)")
shutil.copy(V2, ACTIVE); time.sleep(1)
ingest_pdf_v2(ACTIVE)

print("\nTEST 5 — Queries after v2")
ask_weighted("What vector databases does this guide mention?", show_scores=True)
ask_weighted("What does it say about LLM integration?")

print("\nTEST 6 — Upgrade to v3 (CDC)")
shutil.copy(V3, ACTIVE); time.sleep(1)
ingest_pdf_v2(ACTIVE)

print("\nTEST 7 — Recency verification")
ask_weighted("How should RAG pipelines be evaluated?", show_scores=True)

print("\nTEST 8 — Full stack: recency + SQL Server memory")
m=ConversationMemory()
ask_weighted("What is RAG and what databases does this guide recommend?", memory=m)
ask_weighted("What metrics should I use to evaluate it?", memory=m)
ask_weighted("How does that compare to what you said about databases?", memory=m)

print("\nSUMMARY")
staleness_reg.show()
check_db_stats()
print("\nVerify in SSMS:")
print("  SELECT * FROM rag_chunks ORDER BY ingested_at;")
print("  SELECT * FROM rag_staleness;")
print("  SELECT * FROM rag_sessions;")
print("  SELECT * FROM rag_turns ORDER BY created_at;")

TEST 1 — Full ingest v1

📥 Smart Ingest: rag_guide.pdf
📌 Status: NEW
📄 Step 1/3  Extracting...
   ✅ 1 pages | 540 chars
✂️  Step 2/3  Chunking...
   ✅ 1 chunks
🧠 Step 3/3  Embedding via gemini-embedding-001...
  Embedding chunk 1/1...
  Embedded 1 chunks

✅ Full ingest complete! 1 chunks in rag_chunks
   Hash: 2478ec813ca61d9496b6d855... | Ingested: 2026-04-27T06:48:24
  rag_guide.pdf | v1 | 2026-04-27 06:48:25 | 1 chunks | 2478ec813ca61d9496b6...

TEST 2 — Staleness check (unchanged)

📥 Smart Ingest: rag_guide.pdf
📌 Status: UNCHANGED
✅ Already up to date (v1). Use force=True to re-ingest.

TEST 3 — Queries against v1

 What vector databases does this guide mention?
 alpha=0.85

📊 Score breakdown (α=0.85 cosine | 0.15 recency):
     #    chunk    cosine   recency   blended
   ──────────────────────────────────────────────────
     1  #    0    0.7536       1.0    0.7906
     2  #   49    0.6297       1.0    0.6852
     3  #   48    0.6217       1.0    0.6784
     4  #   52    0.6215   